# 🛡️ AIForensicNet V4 Hybrid + Transfer Learning (Custom Forensic + Pretrained ConvNeXt-Tiny)
### Dataset: `OwensLab/CommunityForensics` + `defactodataset/defactoinpainting`

In [ ]:
# @title 🛠️ 1. Persiapan Lingkungan & Hardware Acceleration
import os, io, sys, shutil, time, random, subprocess, importlib.util, warnings
warnings.filterwarnings('ignore')

for pkg in ('pyarrow', 'huggingface_hub', 'kagglehub'):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
warnings.simplefilter('ignore', Image.DecompressionBombWarning)
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, roc_curve

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__} | Device: {device}' + (f' ({torch.cuda.get_device_name(0)})' if torch.cuda.is_available() else ''))

## 📦 2. Akuisisi Dataset: CommunityForensics (CVPR 2025) + DEFACTO Inpainting
- **Kelas `1` (`Real` - 6.000 sampel):** Citra fotografi/seni otentik dari `OwensLab/CommunityForensics`.
- **Kelas `0` (`AI / Manipulated` - 6.000 sampel):**
  - `3.000` citra AI Generatif dari `OwensLab/CommunityForensics`
  - `3.000` citra Manipulasi Inpainting (Object Removal / Region Filling) dari `defactodataset/defactoinpainting` dengan *Mask-Guided Cropping* (`inpaint_mask` / `probe_mask`).

In [ ]:
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download
import kagglehub

VALID_EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff')
CF_REAL_SAMPLES = 6000       # 6.000 Real untuk mengimbangi 3.000 CF AI + 3.000 DEFACTO Inpainting
CF_AI_SAMPLES = 3000         # 3.000 AI Generatif dari CommunityForensics
DEFACTO_INPAINT_SAMPLES = 3000  # 3.000 Inpainting dari defactodataset/defactoinpainting

def load_community_forensics(n_real=6000, n_ai=3000):
    print(f'[*] Menyiapkan OwensLab/CommunityForensics ({n_real} Real, {n_ai} AI)...')
    CACHE_DIR = 'data/cf_cache'
    os.makedirs(f'{CACHE_DIR}/ai', exist_ok=True)
    os.makedirs(f'{CACHE_DIR}/real', exist_ok=True)

    existing_ai = sorted([os.path.join(f'{CACHE_DIR}/ai', f) for f in os.listdir(f'{CACHE_DIR}/ai') if f.lower().endswith(VALID_EXTS)])
    existing_real = sorted([os.path.join(f'{CACHE_DIR}/real', f) for f in os.listdir(f'{CACHE_DIR}/real') if f.lower().endswith(VALID_EXTS)])
    count_ai, count_real = len(existing_ai), len(existing_real)

    if count_ai < n_ai or count_real < n_real:
        start_shard = min(5, count_real // 4100)
        shards = [f'data/publicEval/HFCF_PublicEval_{i}.parquet' for i in range(start_shard, 6)]
        for shard_path in shards:
            if count_ai >= n_ai and count_real >= n_real:
                break
            print(f'[*] Mengunduh & mengekstrak shard: {shard_path}...')
            try:
                local_pq = hf_hub_download(repo_id='OwensLab/CommunityForensics', filename=shard_path, repo_type='dataset')
                pf = pq.ParquetFile(local_pq)
                rg_indices = list(range(pf.num_row_groups))
                random.shuffle(rg_indices)
                for rg_idx in rg_indices:
                    if count_ai >= n_ai and count_real >= n_real:
                        break
                    b_dict = pf.read_row_group(rg_idx).to_pydict()
                    labels = b_dict.get('label', [])
                    images = b_dict.get('image_data', b_dict.get('image', []))
                    paired = list(zip(labels, images))
                    random.shuffle(paired)
                    for lbl_val, img_data in paired:
                        if count_ai >= n_ai and count_real >= n_real:
                            break
                        is_real = (int(lbl_val) == 0)
                        if (is_real and count_real >= n_real) or ((not is_real) and count_ai >= n_ai):
                            continue
                        try:
                            if isinstance(img_data, Image.Image):
                                img = img_data.convert('RGB')
                            elif isinstance(img_data, (bytes, bytearray)):
                                img = Image.open(io.BytesIO(img_data)).convert('RGB')
                            elif isinstance(img_data, dict) and img_data.get('bytes'):
                                img = Image.open(io.BytesIO(img_data['bytes'])).convert('RGB')
                            else:
                                continue
                        except Exception:
                            continue
                        if max(img.size) > 2048:
                            img.thumbnail((2048, 2048), Image.Resampling.BILINEAR)
                        rand_q = random.randint(75, 98)
                        if is_real:
                            save_p = f'{CACHE_DIR}/real/cf_real_{count_real:05d}.jpg'
                            if not os.path.exists(save_p):
                                img.save(save_p, 'JPEG', quality=rand_q)
                            existing_real.append(save_p)
                            count_real += 1
                        else:
                            save_p = f'{CACHE_DIR}/ai/cf_ai_{count_ai:05d}.jpg'
                            if not os.path.exists(save_p):
                                img.save(save_p, 'JPEG', quality=rand_q)
                            existing_ai.append(save_p)
                            count_ai += 1
                if os.path.exists(local_pq):
                    os.remove(local_pq)
            except Exception as err:
                print(f'[!] Gagal memproses {shard_path}: {err}')

    records = []
    for p in existing_ai[:n_ai]:
        records.append({'image_path': p, 'label': 0, 'label_name': 'AI / Manipulated', 'subset': 'CommunityForensics (AI)'})
    for p in existing_real[:n_real]:
        records.append({'image_path': p, 'label': 1, 'label_name': 'Real', 'subset': 'CommunityForensics (Real)'})
    return records

def load_defacto_inpainting(n_samples=3000):
    CACHE_SUB = 'data/defacto_cache/inpainting_full'
    os.makedirs(CACHE_SUB, exist_ok=True)
    existing = sorted([os.path.join(CACHE_SUB, f) for f in os.listdir(CACHE_SUB) if f.lower().endswith(VALID_EXTS)])

    if len(existing) < n_samples:
        print(f'[*] Mengunduh Kaggle defactodataset/defactoinpainting (target: {n_samples})...')
        try:
            dl_path = kagglehub.dataset_download('defactodataset/defactoinpainting')
            candidates = []
            skip_keywords = ('mask', 'annot', 'graph', 'probe', 'donor')
            for root, _, files in os.walk(dl_path):
                root_lower = root.lower()
                if any(k in root_lower for k in skip_keywords):
                    continue
                for f in files:
                    if f.lower().endswith(VALID_EXTS) and not any(k in f.lower() for k in skip_keywords):
                        candidates.append(os.path.join(root, f))
            random.shuffle(candidates)
            count = len(existing)
            for src_p in tqdm(candidates, desc='Ekstrak DEFACTO Inpainting (Utuh)', leave=False):
                if count >= n_samples:
                    break
                try:
                    img = Image.open(src_p)
                    if img.mode in ('1', 'L', 'I', 'F'):
                        continue
                    img = img.convert('RGB')
                    if max(img.size) > 2048:
                        img.thumbnail((2048, 2048), Image.Resampling.BILINEAR)
                    save_p = f'{CACHE_SUB}/defacto_inpainting_{count:05d}.jpg'
                    if not os.path.exists(save_p):
                        img.save(save_p, 'JPEG', quality=random.randint(75, 98))
                    existing.append(save_p)
                    count += 1
                except Exception:
                    continue
            if os.path.exists(dl_path):
                shutil.rmtree(dl_path, ignore_errors=True)
        except Exception as err:
            print(f'[!] Gagal mengunduh defactoinpainting: {err}')

    return [{'image_path': p, 'label': 0, 'label_name': 'AI / Manipulated', 'subset': 'DEFACTO (Inpainting)'} for p in existing[:n_samples]]

all_records = load_community_forensics(CF_REAL_SAMPLES, CF_AI_SAMPLES) + load_defacto_inpainting(DEFACTO_INPAINT_SAMPLES)
df_all = pd.DataFrame(all_records)
df_ai = df_all[df_all['label'] == 0]
df_real = df_all[df_all['label'] == 1]
n_bal = min(len(df_ai), len(df_real))

df_dataset = pd.concat([
    df_ai.sample(n=n_bal, random_state=42),
    df_real.sample(n=n_bal, random_state=42)
]).sample(frac=1.0, random_state=42).reset_index(drop=True)

print(df_dataset['subset'].value_counts())
print(f'Total Dataset Seimbang: {len(df_dataset):,} citra')

## 🔀 3. Stratified Split (80:10:10) & Native-Resolution DataLoader

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 64

train_df, temp_df = train_test_split(df_dataset, test_size=0.20, random_state=42, stratify=df_dataset['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['label'])
train_df, val_df, test_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

class NativeResolutionCrop:
    def __init__(self, size=224, is_train=True):
        self.size, self.is_train = size, is_train
    def __call__(self, img):
        w, h = img.size
        if w < self.size or h < self.size:
            scale = max(self.size / w, self.size / h)
            img = img.resize((int(round(w * scale)), int(round(h * scale))), Image.Resampling.BILINEAR)
            w, h = img.size
        left = random.randint(0, w - self.size) if self.is_train else (w - self.size) // 2
        top = random.randint(0, h - self.size) if self.is_train else (h - self.size) // 2
        return img.crop((left, top, left + self.size, top + self.size))

class RandomJPEGCompression:
    def __init__(self, quality_range=(70, 95), p=0.4):
        self.quality_range, self.p = quality_range, p
    def __call__(self, img):
        if random.random() < self.p:
            buf = io.BytesIO()
            img.save(buf, format='JPEG', quality=random.randint(*self.quality_range))
            buf.seek(0)
            return Image.open(buf).convert('RGB')
        return img

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    RandomJPEGCompression(quality_range=(70, 95), p=0.4),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class ForensicDataset(Dataset):
    def __init__(self, df, transform):
        self.paths, self.labels, self.transform = df['image_path'].tolist(), df['label'].tolist(), transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (128, 128, 128))
        return self.transform(img), torch.tensor(self.labels[idx], dtype=torch.long)

train_loader = DataLoader(ForensicDataset(train_df, train_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(ForensicDataset(val_df, eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)
test_loader = DataLoader(ForensicDataset(test_df, eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)
print(f'Split -> Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

## 🧠 4. Arsitektur Model Buatan + Transfer Learning (`AIForensicNetV4HybridTL`)

In [ ]:
class SqueezeExcitation(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        reduced = max(8, channels // reduction)
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, reduced, bias=False), nn.GELU(),
            nn.Linear(reduced, channels, bias=False), nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.shape
        return x * self.fc(x).view(b, c, 1, 1)

class ForensicBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.dwconv = nn.Conv2d(in_c, in_c, kernel_size=7, stride=stride, padding=3, groups=in_c, bias=False)
        self.norm = nn.BatchNorm2d(in_c)
        self.pwconv1 = nn.Conv2d(in_c, out_c * 2, kernel_size=1)
        self.act = nn.GELU()
        self.pwconv2 = nn.Conv2d(out_c * 2, out_c, kernel_size=1)
        self.se = SqueezeExcitation(out_c)
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=1, stride=stride, bias=False),
            nn.BatchNorm2d(out_c)
        ) if stride != 1 or in_c != out_c else nn.Identity()
    def forward(self, x):
        return self.se(self.pwconv2(self.act(self.pwconv1(self.norm(self.dwconv(x)))))) + self.shortcut(x)

class NPRModule(nn.Module):
    def forward(self, x):
        down = F.interpolate(x, scale_factor=0.5, mode='nearest')
        return x - F.interpolate(down, size=x.shape[2:], mode='nearest')

class FFTSpectralBranch(nn.Module):
    def __init__(self, in_channels=3, out_channels=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.GELU(),
            nn.Conv2d(16, out_channels, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(out_channels), nn.GELU()
        )
    def forward(self, x):
        fft = torch.fft.rfft2(x.float(), norm='ortho')
        mag = F.interpolate(torch.log(torch.abs(fft) + 1e-6), size=x.shape[2:], mode='bilinear', align_corners=False).to(x.dtype)
        return self.net(mag)

class MultiScaleForensicFilterBankV4(nn.Module):
    def __init__(self):
        super().__init__()
        lap3 = torch.tensor([[-1., -1., -1.], [-1., 8., -1.], [-1., -1., -1.]])
        lap5 = torch.tensor([[0., 0., -1., 0., 0.], [0., -1., -2., -1., 0.], [-1., -2., 16., -2., -1.], [0., -1., -2., -1., 0.], [0., 0., -1., 0., 0.]])
        dog = torch.tensor([[-1., -3., -4., -3., -1.], [-3., 0., 6., 0., -3.], [-4., 6., 20., 6., -4.], [-3., 0., 6., 0., -3.], [-1., -3., -4., -3., -1.]]) / 16.0
        sobel_h = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]])
        sobel_v = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])
        self.register_buffer('k3', torch.stack([lap3, sobel_h, sobel_v]).unsqueeze(1).repeat(3, 1, 1, 1))
        self.register_buffer('k5', torch.stack([lap5, dog]).unsqueeze(1).repeat(3, 1, 1, 1))
        self.npr = NPRModule()
    def forward(self, x):
        f3 = F.conv2d(x, self.k3, padding=1, groups=3)
        f5 = F.conv2d(x, self.k5, padding=2, groups=3)
        smag = torch.sqrt(f3[:, 1::3] ** 2 + f3[:, 2::3] ** 2 + 1e-6)
        return torch.cat([f3[:, 0::3], f5[:, 0::2], f5[:, 1::2], smag, self.npr(x)], dim=1)

class MultiStatPool2d(nn.Module):
    def forward(self, x):
        mean = torch.mean(x, dim=(2, 3))
        var = torch.var(x, dim=(2, 3), unbiased=False)
        return torch.cat([mean, torch.sqrt(var + 1e-6)], dim=1)

class AIForensicNetV4HybridTL(nn.Module):
    """
    Model Buatan (Custom Forensic + 2D-FFT Stream) + Transfer Learning (Pretrained ConvNeXt-Tiny ImageNet-1K).
    Menggabungkan kepekaan residu frekuensi mikro (NPR, Laplacian, DoG, FFT) dengan representasi semantik
    pretrained melalui Gated Cross-Attention Fusion.
    """
    def __init__(self, num_classes=2, pretrained=True, dropout_p=0.3):
        super().__init__()
        # 1. Cabang Transfer Learning (Pretrained ConvNeXt-Tiny)
        weights = models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None
        backbone = models.convnext_tiny(weights=weights)
        self.tl_features = backbone.features
        self.tl_avgpool = backbone.avgpool
        self.tl_norm = backbone.classifier[0]  # LayerNorm2d(768)

        # 2. Cabang Model Buatan (Custom Forensic Filter Bank 15-ch + 2D-FFT + ForensicBlock)
        self.forensic_bank = MultiScaleForensicFilterBankV4()
        self.forensic_stem = nn.Sequential(nn.Conv2d(15, 32, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(32), nn.GELU())
        self.spectral_branch = FFTSpectralBranch(3, 32)
        self.custom_fusion = nn.Sequential(nn.Conv2d(64, 64, 1, bias=False), nn.BatchNorm2d(64), nn.GELU())
        self.c_stage1 = nn.Sequential(ForensicBlock(64, 64), ForensicBlock(64, 64))
        self.c_stage2 = nn.Sequential(ForensicBlock(64, 128, stride=2), ForensicBlock(128, 128))
        self.c_stage3 = nn.Sequential(ForensicBlock(128, 256, stride=2), ForensicBlock(256, 256))
        self.c_pool = MultiStatPool2d()  # 512-D

        # 3. Gated Hybrid Fusion Head
        self.proj_tl = nn.Sequential(nn.Linear(768, 512), nn.LayerNorm(512), nn.GELU())
        self.proj_custom = nn.Sequential(nn.Linear(512, 512), nn.LayerNorm(512), nn.GELU())
        self.gate = nn.Sequential(nn.Linear(1024, 512), nn.Sigmoid())
        self.head = nn.Sequential(
            nn.Dropout(p=dropout_p), nn.Linear(512, 128),
            nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(p=0.15), nn.Linear(128, num_classes)
        )

    def forward(self, x):
        f_tl = self.tl_norm(self.tl_avgpool(self.tl_features(x))).flatten(1)
        v_tl = self.proj_tl(f_tl)

        f_c = self.custom_fusion(torch.cat([self.forensic_stem(self.forensic_bank(x)), self.spectral_branch(x)], dim=1))
        f_c = self.c_stage3(self.c_stage2(self.c_stage1(f_c)))
        v_c = self.proj_custom(self.c_pool(f_c))

        g = self.gate(torch.cat([v_tl, v_c], dim=1))
        return self.head(g * v_tl + (1.0 - g) * v_c)

model = AIForensicNetV4HybridTL(num_classes=2, pretrained=True).to(device)
BEST_MODEL_PATH = 'best_aiforensicnet_v4_cf_inpainting_hybrid_tl.pth'
LEARNING_RATE = 1.5e-4  # Differential/fine-tuning LR untuk stabilitas pretrained backbone + custom branch
print(f'Model Buatan + Transfer Learning Siap! Total Parameter: {sum(p.numel() for p in model.parameters()):,}')

## 🔍 5. Visualisasi Ekstraksi Sinyal Forensik (Laplacian, DoG, NPR & 2D-FFT)

In [ ]:
# @title 🔍 Visualisasi Sinyal Forensik pada Sampel Real vs AI / Inpainting
sample_ai_row = df_dataset[df_dataset['label'] == 0].iloc[0]
sample_real_row = df_dataset[df_dataset['label'] == 1].iloc[0]

def inspect_forensic_signals(pil_img, title_prefix='Sample'):
    model.eval()
    t_img = eval_transform(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        f_bank = model.forensic_bank(t_img)[0]
        lap3_vis = f_bank[0:3].mean(dim=0).cpu().numpy()
        dog_vis = f_bank[6:9].mean(dim=0).cpu().numpy()
        npr_vis = f_bank[12:15].mean(dim=0).cpu().numpy()
        fft_mag = torch.log(torch.abs(torch.fft.fftshift(torch.fft.fft2(t_img.float(), norm='ortho'))) + 1e-6)[0].mean(dim=0).cpu().numpy()

    fig, axes = plt.subplots(1, 5, figsize=(17, 3.3))
    axes[0].imshow(pil_img.resize((IMG_SIZE, IMG_SIZE)))
    axes[0].set_title(title_prefix, fontweight='bold')
    axes[1].imshow(lap3_vis, cmap='gray'); axes[1].set_title('Laplacian 3x3')
    axes[2].imshow(dog_vis, cmap='inferno'); axes[2].set_title('DoG 5x5')
    axes[3].imshow(npr_vis, cmap='coolwarm'); axes[3].set_title('NPR Residual (CVPR 24)')
    axes[4].imshow(fft_mag, cmap='viridis'); axes[4].set_title('2D-FFT Spectrum')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

inspect_forensic_signals(Image.open(sample_real_row['image_path']).convert('RGB'), 'REAL (Authentic)')
inspect_forensic_signals(Image.open(sample_ai_row['image_path']).convert('RGB'), f"FAKE ({sample_ai_row['subset']})")

## 🚀 6. Pelatihan AMP FP16 + Focal Loss, Kurva Metrik, Confusion Matrix & ROC Curve

In [ ]:
# @title 🚀 6. Pelatihan AMP FP16 + Focal Loss, Kurva Metrik, Confusion Matrix & ROC Curve
EPOCHS = 20
PATIENCE = 5
WEIGHT_DECAY = 1e-4

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.gamma, self.label_smoothing = gamma, label_smoothing
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, label_smoothing=self.label_smoothing, reduction='none')
        return (((1.0 - torch.exp(-ce)) ** self.gamma) * ce).mean()

criterion = FocalLoss(gamma=2.0, label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
use_amp = (device.type == 'cuda')
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': []}
best_val_f1, no_improve = 0.0, 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, train_preds, train_targets = 0.0, [], []
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch:02d}/{EPOCHS:02d} [Train]', leave=False):
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)
        train_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        train_targets.extend(labels.cpu().numpy())

    epoch_train_loss = running_loss / len(train_df)
    epoch_train_acc = (np.array(train_preds) == np.array(train_targets)).mean() * 100

    model.eval()
    val_loss, val_preds, val_targets = 0.0, [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=use_amp):
                outputs = model(images)
                loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            val_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            val_targets.extend(labels.cpu().numpy())

    epoch_val_loss = val_loss / len(val_df)
    epoch_val_acc = (np.array(val_preds) == np.array(val_targets)).mean() * 100
    val_f1 = f1_score(val_targets, val_preds, average='macro') * 100
    scheduler.step()

    history['train_loss'].append(epoch_train_loss)
    history['val_loss'].append(epoch_val_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_acc'].append(epoch_val_acc)
    history['val_f1'].append(val_f1)

    saved = ''
    if val_f1 > best_val_f1:
        best_val_f1, no_improve = val_f1, 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        saved = '==> [CHECKPOINT TERSIMPAN]'
    else:
        no_improve += 1
    print(f'Epoch {epoch:02d}/{EPOCHS:02d} | Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.2f}% | Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.2f}% F1: {val_f1:.2f}% {saved}')
    if no_improve >= PATIENCE:
        print(f'Early stopping pada epoch {epoch}.')
        break

# Kurva Training
ep_range = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(ep_range, history['train_loss'], marker='o', label='Train Loss')
axes[0].plot(ep_range, history['val_loss'], marker='s', label='Val Loss')
axes[0].set_title('Kurva Loss (Train vs Validation)', fontweight='bold'); axes[0].grid(True, alpha=0.3); axes[0].legend()
axes[1].plot(ep_range, history['train_acc'], marker='o', label='Train Acc (%)')
axes[1].plot(ep_range, history['val_acc'], marker='s', label='Val Acc (%)')
axes[1].plot(ep_range, history['val_f1'], linestyle='--', marker='^', label='Val Macro F1 (%)')
axes[1].set_title('Kurva Akurasi & Macro F1-Score', fontweight='bold'); axes[1].grid(True, alpha=0.3); axes[1].legend()
plt.tight_layout(); plt.show()

# Evaluasi Test Set + Confusion Matrix + ROC Curve
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()
test_preds, test_targets, test_probs = [], [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Test Set (TTA)'):
        images = images.to(device)
        with torch.cuda.amp.autocast(enabled=use_amp):
            probs = (torch.softmax(model(images), dim=1) + torch.softmax(model(torch.flip(images, dims=[3])), dim=1)) / 2.0
        test_probs.extend(probs[:, 1].cpu().numpy())
        test_preds.extend(probs.argmax(dim=1).cpu().numpy())
        test_targets.extend(labels.numpy())

test_targets, test_preds, test_probs = np.array(test_targets), np.array(test_preds), np.array(test_probs)
test_acc = (test_targets == test_preds).mean() * 100
test_auc = roc_auc_score(test_targets, test_probs) * 100
print(classification_report(test_targets, test_preds, target_names=['AI / Inpainting (0)', 'Real (1)'], digits=4))
print(f'Akurasi Test Set: {test_acc:.2f}% | ROC-AUC: {test_auc:.2f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(test_targets, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred AI/Inpaint (0)', 'Pred Real (1)'],
            yticklabels=['True AI/Inpaint (0)', 'True Real (1)'])
axes[0].set_title(f'Confusion Matrix (Akurasi: {test_acc:.2f}%)', fontweight='bold')

fpr, tpr, _ = roc_curve(test_targets, test_probs)
axes[1].plot(fpr, tpr, color='#2563EB', lw=2.5, label=f'ROC Curve (AUC = {test_auc/100:.4f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[1].set_title('Receiver Operating Characteristic (ROC)', fontweight='bold')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right'); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 💾 8. Auto Unduh Bobot Model untuk `app.py` (Streamlit)
Otomatis mengunduh file checkpoint terbaik ke komputer lokal Anda setelah evaluasi selesai.

In [ ]:
# @title 💾 Auto Unduh Checkpoint Model (.pth)
try:
    from google.colab import files
    if os.path.exists(BEST_MODEL_PATH):
        print(f'[*] Mengunduh otomatis {BEST_MODEL_PATH} ke perangkat Anda...')
        files.download(BEST_MODEL_PATH)
    else:
        print(f'[!] File {BEST_MODEL_PATH} tidak ditemukan!')
except ImportError:
    print(f'[+] Berjalan di lingkungan lokal. Model sudah tersimpan di: {os.path.abspath(BEST_MODEL_PATH)}')